# Monarch · Wan 2.1 — 36s film renderer (free GPU)

**Sirf 2 kaam:** upar `Runtime → Change runtime type → T4 GPU` chuno, phir **Run all** dabao.
Baqi sab automatic hai: 12 scenes render honge, Google Drive me `Monarch-T2V/clips/` me jayenge,
aur main (agent) wahan se utha kar final 36s film bana dunga.

Wan 2.1 T2V-1.3B = 2025 ka asli video model. Yeh GPU par chalta hai, is liye CPU wale
AnimateDiff se bohat aage ka result deta hai.


In [ ]:
import subprocess, sys, torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    print("\nGPU nahi mila. Runtime > Change runtime type > T4 GPU chuno, phir dobara Run all.")
    raise SystemExit("no gpu")
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
!pip -q install "diffusers>=0.32" "transformers>=4.49" accelerate safetensors sentencepiece protobuf ftfy imageio imageio-ffmpeg
print("installed")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib, json
OUT = pathlib.Path('/content/drive/MyDrive/Monarch-T2V/clips')
OUT.mkdir(parents=True, exist_ok=True)
print("output:", OUT)


In [ ]:
import json, urllib.request

RAW = "https://raw.githubusercontent.com/adil-chandio/Monarch-Studio/arena/3d8cfad7-monarch-studio/cloud/t2v_jobs/personality-wan.json"
job = json.loads(urllib.request.urlopen(RAW).read().decode())
print(job["id"], len(job["scenes"]), "scenes")


In [ ]:
import torch
from diffusers import WanPipeline
from diffusers.utils import export_to_video

MODEL = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
pipe = WanPipeline.from_pretrained(MODEL, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()          # T4 (16GB) par fit karne ke liye
pipe.vae.to(torch.float32)               # fp16 VAE black frames deta hai
pipe.vae.enable_slicing()
pipe.set_progress_bar_config(disable=True)
print("Wan loaded")


In [ ]:
import time, json, pathlib

STEPS = 28
results = []
for scene in job["scenes"]:
    clip = OUT / f'{scene["id"]}.mp4'
    if clip.exists() and clip.stat().st_size > 50000:
        print(scene["id"], "already done"); results.append({"scene": scene["id"], "state": "skip"}); continue
    t0 = time.time()
    try:
        with torch.no_grad():
            frames = pipe(
                prompt=f'{scene["prompt"]}, {job["style"]}',
                negative_prompt=job["negative"],
                height=int(job["height"]), width=int(job["width"]),
                num_frames=int(job["frames"]), num_inference_steps=STEPS,
                guidance_scale=float(job["guidance"]),
                generator=torch.Generator("cpu").manual_seed(int(scene["seed"])),
            ).frames[0]
        tmp = pathlib.Path("/content") / f'{scene["id"]}.mp4'
        export_to_video(frames, str(tmp), fps=int(job["raw_fps"]))
        clip.write_bytes(tmp.read_bytes())
        dt = time.time() - t0
        print(scene["id"], "OK", len(frames), "frames", round(dt/60, 1), "min")
        results.append({"scene": scene["id"], "state": "ok", "seconds": round(dt, 1),
                        "bytes": clip.stat().st_size})
    except Exception as exc:
        print(scene["id"], "FAIL", type(exc).__name__, str(exc)[:300])
        results.append({"scene": scene["id"], "state": "fail", "error": str(exc)[:300]})
    (OUT / "receipt.json").write_text(json.dumps(results, indent=2))
print("batch done")


In [ ]:
done = [r for r in results if r["state"] == "ok"]
failed = [r for r in results if r["state"] == "fail"]
print(f"{len(done)} naye clips, {len(failed)} fail")
print("Drive folder:", OUT)
print(json.dumps(failed, indent=2) if failed else "sab theek")
